# 🔮 Fortuna - Financial Assistant Training Pipeline

This notebook trains the Qwen3-0.6B model for the Fortuna iOS financial assistant app.

**What this notebook does:**
1. Downloads and merges datasets (Fortuna synthetic + finance-alpaca)
2. Trains with QAT (Quantization-Aware Training) for mobile deployment
3. Tracks training with Weights & Biases
4. Exports to ExecuTorch format (4-bit and 8-bit)

**Requirements:** Free Google Colab T4 GPU

## 1. Setup & Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install torchao==0.14.0 executorch pytorch_tokenizers
!pip install wandb

## 2. Configuration

In [ ]:
# ============== CONFIGURATION ==============
CONFIG = {
    # Model
    "model_name": "unsloth/Qwen3-0.6B",
    "max_seq_length": 1024,
    
    # Training
    "num_epochs": 3,
    "batch_size": 2,
    "gradient_accumulation": 4,
    "learning_rate": 5e-5,
    "logging_steps": 10,
    "save_steps": 200,
    
    # Dataset
    "use_finance_alpaca": True,  # Set to False to use only Fortuna data
    "finance_alpaca_sample": 15000,  # Max samples from finance-alpaca
    
    # W&B
    "wandb_project": "fortuna-llm",
    "wandb_run_name": "fortuna-qwen3-v1",
    
    # Output
    "output_dir": "./fortuna_output",
    "model_version": "1.0.0",
}

print("✅ Configuration loaded")
print(f"  Model: {CONFIG['model_name']}")
print(f"  Epochs: {CONFIG['num_epochs']}")
print(f"  Use finance-alpaca: {CONFIG['use_finance_alpaca']}")

## 3. Initialize W&B Tracking

In [ ]:
import wandb

# Login to W&B (will prompt for API key)
wandb.login()

# Initialize run
run = wandb.init(
    project=CONFIG["wandb_project"],
    name=CONFIG["wandb_run_name"],
    config=CONFIG,
    tags=["finetuning", "qat", "production", "financial-assistant"],
)

print(f"✅ W&B initialized: {run.url}")

## 4. Load and Prepare Dataset

In [ ]:
from datasets import load_dataset, Dataset
import json
import random

# System prompt for Fortuna
SYSTEM_PROMPT = """You are Fortuna, a friendly AI financial buddy. You help users understand their money and make better decisions.

Guidelines:
- Be casual, warm, and non-judgmental
- Use occasional emojis but don't overdo it
- Give personalized advice based on their specific situation
- If asked about specific investments, remind them you're not a licensed advisor
- Encourage good habits without being preachy
- Keep responses concise (under 150 words usually)"""

def format_example(example):
    """Format example into conversation format."""
    conversations = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ]
    
    # User message
    user_content = example["instruction"]
    if example.get("input") and example["input"].strip():
        user_content += "\n\n" + example["input"]
    
    conversations.append({"role": "user", "content": user_content.strip()})
    conversations.append({"role": "assistant", "content": example["output"]})
    
    return {"conversations": conversations}

print("📥 Loading datasets...")

In [ ]:
# Upload Fortuna dataset (run this cell and upload fortuna_synthetic_dataset.json)
from google.colab import files

print("📤 Please upload fortuna_synthetic_dataset.json")
uploaded = files.upload()

# Load uploaded Fortuna data
fortuna_data = []
for filename in uploaded.keys():
    if filename.endswith('.json'):
        with open(filename, 'r') as f:
            fortuna_data = json.load(f)
        print(f"✅ Loaded {len(fortuna_data)} examples from {filename}")
        break

if not fortuna_data:
    print("⚠️ No Fortuna data uploaded. Using only finance-alpaca.")

In [ ]:
# Load finance-alpaca from HuggingFace
all_examples = []

if CONFIG["use_finance_alpaca"]:
    print("📥 Loading finance-alpaca from HuggingFace...")
    finance_alpaca = load_dataset("gbharti/finance-alpaca", split="train")
    
    # Convert to list
    fa_examples = [{"instruction": x["instruction"], "input": x["input"], "output": x["output"]} 
                   for x in finance_alpaca]
    
    # Filter relevant examples
    relevant_keywords = [
        "invest", "stock", "bond", "fund", "etf", "401k", "ira", "roth",
        "budget", "save", "saving", "debt", "loan", "credit", "mortgage",
        "interest", "compound", "dividend", "portfolio", "retirement",
        "tax", "income", "expense", "money", "financial", "wealth",
    ]
    
    fa_filtered = [ex for ex in fa_examples 
                   if any(kw in (ex["instruction"] + ex["output"]).lower() for kw in relevant_keywords)
                   and len(ex["output"]) < 3000]
    
    # Sample if needed
    if len(fa_filtered) > CONFIG["finance_alpaca_sample"]:
        fa_filtered = random.sample(fa_filtered, CONFIG["finance_alpaca_sample"])
    
    all_examples.extend(fa_filtered)
    print(f"  Added {len(fa_filtered)} finance-alpaca examples")

# Add Fortuna data
if fortuna_data:
    all_examples.extend(fortuna_data)
    print(f"  Added {len(fortuna_data)} Fortuna examples")

# Shuffle
random.shuffle(all_examples)
print(f"\n📊 Total training examples: {len(all_examples)}")

In [ ]:
# Format dataset for training
formatted_examples = [format_example(ex) for ex in all_examples]

# Create HuggingFace Dataset
train_dataset = Dataset.from_list(formatted_examples)

# Log dataset info to W&B
wandb.log({
    "dataset/total_examples": len(train_dataset),
    "dataset/fortuna_examples": len(fortuna_data) if fortuna_data else 0,
    "dataset/finance_alpaca_examples": len(all_examples) - (len(fortuna_data) if fortuna_data else 0),
})

# Log sample examples as table
sample_table = wandb.Table(columns=["user_message", "assistant_response"])
for ex in formatted_examples[:10]:
    sample_table.add_data(
        ex["conversations"][1]["content"][:500],
        ex["conversations"][2]["content"][:500]
    )
wandb.log({"dataset/samples": sample_table})

print(f"✅ Dataset prepared: {len(train_dataset)} examples")
print(f"\n📝 Sample conversation:")
print(f"User: {train_dataset[0]['conversations'][1]['content'][:200]}...")
print(f"Assistant: {train_dataset[0]['conversations'][2]['content'][:200]}...")

## 5. Load Model with QAT

In [ ]:
from unsloth import FastLanguageModel
import torch

print("🔄 Loading model with QAT for phone deployment...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    full_finetuning=True,
    qat_scheme="phone-deployment",
)

# Log model info
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

wandb.log({
    "model/total_parameters": total_params,
    "model/trainable_parameters": trainable_params,
    "model/parameters_millions": total_params / 1e6,
})

print(f"✅ Model loaded!")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

## 6. Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation"],
    learning_rate=CONFIG["learning_rate"],
    logging_steps=CONFIG["logging_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=3,
    optim="adamw_8bit",
    weight_decay=0.001,
    lr_scheduler_type="linear",
    warmup_steps=50,
    report_to="wandb",
    run_name=CONFIG["wandb_run_name"],
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=training_args,
)

print("🚀 Starting training...")

In [ ]:
# Train!
trainer.train()

In [ ]:
# Log final metrics
if trainer.state.log_history:
    final_loss = trainer.state.log_history[-1].get("loss", "N/A")
    wandb.log({"training/final_loss": final_loss})
    print(f"\n✅ Training complete! Final loss: {final_loss}")

## 7. Test the Model

In [ ]:
# Test prompts
test_prompts = [
    "Should I buy a PS5 if I have $2000 in savings and $500 in credit card debt?",
    "What's compound interest? Explain it simply.",
    "I make $5000/month and have $15k in student loans. What's my financial vibe?",
    "How do I start investing with only $100?",
]

print("🧪 Testing model responses:\n")
print("=" * 60)

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the assistant's response
    response = response.split("assistant")[-1].strip()
    
    print(f"\n💬 User: {prompt}")
    print(f"\n🔮 Fortuna: {response[:500]}...")
    print("\n" + "-" * 60)

## 8. Save Model

In [ ]:
# Save for ExecuTorch export
print("💾 Saving model for ExecuTorch...")
model.save_pretrained_torchao("fortuna_phone_model", tokenizer=tokenizer)

# Log model as W&B artifact
model_artifact = wandb.Artifact(
    name=f"fortuna-model-{CONFIG['model_version']}",
    type="model",
    metadata={
        "base_model": CONFIG["model_name"],
        "training_epochs": CONFIG["num_epochs"],
        "version": CONFIG["model_version"],
        "ready_for_executorch": True,
    }
)
model_artifact.add_dir("fortuna_phone_model")
wandb.log_artifact(model_artifact)

print("✅ Model saved!")

## 9. Export to ExecuTorch

In [ ]:
# Convert weights for ExecuTorch
print("🔄 Converting weights for ExecuTorch...")
!python -m executorch.examples.models.qwen3.convert_weights \
    "fortuna_phone_model" pytorch_model_converted.bin

In [ ]:
# Download model config
!curl -L -o 0.6B_config.json https://raw.githubusercontent.com/pytorch/executorch/main/examples/models/qwen3/config/0_6b_config.json

In [ ]:
# Export 8-bit version
print("📦 Exporting 8-bit ExecuTorch model...")
!python -m executorch.examples.models.llama.export_llama \
    --model "qwen3_0_6b" \
    --checkpoint pytorch_model_converted.bin \
    --params 0.6B_config.json \
    --output_name fortuna_int8.pte \
    -kv \
    --use_sdpa_with_kv_cache \
    -X \
    --xnnpack-extended-ops \
    --max_context_length 1024 \
    --max_seq_length 128 \
    --dtype fp32 \
    --pt2e_quantize xnnpack_dynamic \
    --metadata '{"get_bos_id":199999, "get_eos_ids":[200020,199999]}'

In [ ]:
# Export 4-bit version
print("📦 Exporting 4-bit ExecuTorch model...")
!python -m executorch.examples.models.llama.export_llama \
    --model "qwen3_0_6b" \
    --checkpoint pytorch_model_converted.bin \
    --params 0.6B_config.json \
    --output_name fortuna_int4.pte \
    -kv \
    --use_sdpa_with_kv_cache \
    -X \
    --xnnpack-extended-ops \
    --max_context_length 1024 \
    --max_seq_length 128 \
    --dtype fp32 \
    --pt2e_quantize xnnpack_dynamic_qc4 \
    --metadata '{"get_bos_id":199999, "get_eos_ids":[200020,199999]}'

In [ ]:
# Check exported files
import os

files_info = []
for f in ["fortuna_int8.pte", "fortuna_int4.pte"]:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / (1024 * 1024)
        files_info.append({"file": f, "size_mb": size_mb})
        print(f"✅ {f}: {size_mb:.1f} MB")

# Log to W&B
wandb.log({
    "export/int8_size_mb": files_info[0]["size_mb"] if len(files_info) > 0 else 0,
    "export/int4_size_mb": files_info[1]["size_mb"] if len(files_info) > 1 else 0,
})

## 10. Download Models

In [ ]:
from google.colab import files

# Download the exported models
print("📥 Downloading exported models...")
files.download("fortuna_int8.pte")
files.download("fortuna_int4.pte")

In [ ]:
# Also save tokenizer for iOS app
tokenizer.save_pretrained("fortuna_tokenizer")

# Zip tokenizer
!zip -r fortuna_tokenizer.zip fortuna_tokenizer/
files.download("fortuna_tokenizer.zip")

## 11. Finish W&B Run

In [ ]:
# Log final summary
wandb.summary["training_complete"] = True
wandb.summary["models_exported"] = ["int4", "int8"]

# Finish run
wandb.finish()

print("\n" + "=" * 60)
print("✅ TRAINING PIPELINE COMPLETE!")
print("=" * 60)
print("\n📦 Exported models:")
print("  - fortuna_int8.pte (8-bit, better quality)")
print("  - fortuna_int4.pte (4-bit, smaller size)")
print("\n📱 Next steps:")
print("  1. Add .pte files to your iOS Xcode project")
print("  2. Use ExecuTorch runtime to load and run")
print("  3. See ExecuTorch iOS docs for integration")

---

## 📚 Next Steps for iOS Integration

After downloading the `.pte` files:

1. **Add to Xcode Project**
   - Drag `fortuna_int8.pte` (or `int4`) into your Xcode project
   - Ensure "Copy items if needed" is checked

2. **Add ExecuTorch Framework**
   - Follow ExecuTorch iOS setup guide
   - Add the ExecuTorch pod or framework

3. **Load Model in Swift**
```swift
import ExecuTorch

let modelPath = Bundle.main.path(forResource: "fortuna_int8", ofType: "pte")!
let module = try Module(modelPath: modelPath)
```

4. **Run Inference**
```swift
let output = try module.forward(inputs: [tokenizedInput])
```

See the ExecuTorch documentation for full iOS integration details!